In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.10 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3132
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


SEED = 3132. The split and the folds use it, and so does every random model.

In [3]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   # members of voting and stacking in Parts 3-4

The models are defined once here and not changed later. When I fit one more than once I use `clone`.

## Part 1 — Bagging and out-of-bag estimates

In [4]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask

In [5]:
print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))

[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [6]:
bag.fit(X_train, y_train)
n = len(y_train)

oob_share = np.mean([oob_mask(s, n).mean() for s in bag.estimators_samples_])
print(f"mean OOB share:      {oob_share:.4f}")
print(f"mean in-bag share:   {1 - oob_share:.4f}")
print(f"(1 - 1/n)^n:         {(1 - 1/n) ** n:.4f}")
print(f"1 - (1 - 1/n)^n:     {1 - (1 - 1/n) ** n:.4f}")

mean OOB share:      0.3683
mean in-bag share:   0.6317
(1 - 1/n)^n:         0.3678
1 - (1 - 1/n)^n:     0.6322


In [7]:
votes = np.zeros((n, 2), dtype=int)
for est, samples in zip(bag.estimators_, bag.estimators_samples_):
    mask = oob_mask(samples, n)
    pred = est.predict(X_train[mask])
    votes[np.where(mask)[0], pred] += 1

oob_pred = votes.argmax(axis=1)      # tie goes to class 0
my_oob_acc = (oob_pred == y_train).mean()
print(f"my OOB accuracy:  {my_oob_acc:.4f}")
print(f"bag.oob_score_:   {bag.oob_score_:.4f}")

my OOB accuracy:  0.9369
bag.oob_score_:   0.9369


On average a tree misses 36.8% of the training rows (OOB) and sees 63.2% of them. The formula (1 - 1/n)^n gives 36.8% too, so the bootstrap behaves as expected: each tree has about a third of the rows that it never saw.

My own OOB accuracy is 0.9369 and `bag.oob_score_` is 0.9369. They are the same, so my vote counting does what scikit-learn does.

In [8]:
from sklearn.model_selection import cross_validate

models = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}
scores = {name: cross_validate(m, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
          for name, m in models.items()}

table = pd.DataFrame(scores).T
table.columns = [f"fold {i+1}" for i in range(5)]
table["mean"] = table.iloc[:, :5].mean(axis=1)
table["std"] = table.iloc[:, :5].std(axis=1, ddof=0)
table.round(4)

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
tree,0.9002,0.9017,0.8859,0.9079,0.9048,0.9001,0.0076
bag,0.9334,0.9287,0.9350,0.9302,0.9524,0.9359,0.0085
rf,0.9366,0.9477,0.9572,0.9492,0.9556,0.9493,0.0073
lr,0.9382,0.9382,0.9303,0.9365,0.9492,0.9385,0.0061
bag_lr,0.9382,0.9414,0.9303,0.9349,0.9524,0.9394,0.0074


In [9]:
def paired(a, b):
    d = scores[a] - scores[b]
    verdict = "inside spread" if abs(d.mean()) <= d.std() else "outside spread"
    print(f"{a} - {b}: {np.round(d, 4)}  mean={d.mean():.4f}  std={d.std():.4f}  -> {verdict}")

paired("bag", "tree")
paired("rf", "tree")
paired("bag_lr", "lr")

bag - tree: [0.0333 0.0269 0.0491 0.0222 0.0476]  mean=0.0358  std=0.0108  -> outside spread
rf - tree: [0.0365 0.046  0.0713 0.0413 0.0508]  mean=0.0492  std=0.0121  -> outside spread
bag_lr - lr: [ 0.      0.0032  0.     -0.0016  0.0032]  mean=0.0010  std=0.0019  -> inside spread


In [10]:
rf.fit(X_train, y_train)
p = rf.score(X_test, y_test)
se = np.sqrt(p * (1 - p) / len(y_test))

print(f"rf oob_score_:        {rf.oob_score_:.4f}")
print(f"rf CV mean:           {scores['rf'].mean():.4f}  (fold std {scores['rf'].std():.4f})")
print(f"rf test accuracy:     {p:.4f}  (standard error {se:.4f})")
print(f"|oob - CV mean|:      {abs(rf.oob_score_ - scores['rf'].mean()):.4f}")

rf oob_score_:        0.9477
rf CV mean:           0.9493  (fold std 0.0073)
rf test accuracy:     0.9420  (standard error 0.0072)
|oob - CV mean|:      0.0016


**(a) Does bagging help?**
For the tree, yes. Bagging raised the CV accuracy from 0.900 to 0.936, and the gain is positive on all five folds (mean +0.036, std 0.011), so it is clearly larger than the fold spread. A single deep tree has high variance: it memorises the training rows. Averaging 100 trees trained on different bootstrap samples smooths that out.
For logistic regression, no. The change is +0.001 with std 0.002, which is inside the spread, so my data cannot tell it apart from zero. Logistic regression is stable: a bootstrap sample gives almost the same model, so there is little variance for bagging to remove.

**(b) Is OOB a fair stand-in for CV?**
Yes. The forest's `oob_score_` is 0.9477 and its CV mean is 0.9493. The difference, 0.0016, is much smaller than the fold std of 0.0073, so they agree. The test accuracy is 0.9420 with a standard error of 0.0072. It is a little lower than both, but the gap is about one standard error, so all three numbers tell the same story.